# Module 12: Review Filing Evidence and Apply Sample Decisions

目标：把人工审核落实到数据和配对样本，保留证据、排除原因与修正前后数量。
仍为 8 个 Code Cells：中文逐格讲解，英文代码、批注、字段和文件名，采用普通循环和 pandas；不写自定义分析函数。
本模块不替你把未阅读的年报标为已审核，也不自动下载或生成企业经营原因。

**请采用两轮运行：**
1. 第一次可以运行全部 8 格，确认审核工作表和样本重建正常。全部 pending 时，仍应是毛利 401 对、现金流 490 对；这不表示人工审核已经完成。
2. 随后阅读年报，编辑 Cell 2 创建的四份 CSV。保存后从 Cell 3 重跑至 Cell 8，让决定实际生效。
3. 审核有新修改，就重跑 Cell 3–8；不要只运行 Cell 8，它只会保存 Kernel 中旧变量。
4. 导出表保存在 `work/data/processed/module12_reviewed/`，其中默认模式仍明确标为 provisional。目录名称不代表审核完成。
5. 原 Module 8–11 数据和人工文件不覆盖。Module 12 工作表只在不存在时创建，再次运行不会抹掉你的备注。

默认 `REPORT_SAMPLE = "provisional"`：保留 pending 记录、应用有证据的补值与排除，产生审核后的暂定样本。
可改为 `"reviewed_only"`：公司资格已批准纳入，且上期、本期、下一期三个年度均已审核的配对才可能进入。
此模式还要求每种结果至少 5 个经过审核且数值可用的年度。当前全部 pending 时导出空表是正常结果；不能拿空表继续跑回归。
小规模已审核子集也不保证足够支持 Module 11 的公司／年份模型。

**这周末优先做什么？**
先核对 7 个案例所涉及的 6 家公司资格，及其上期／本期／下一期年度。Cell 2 把相关年度排到审核表前面。
再按影响大小处理缺失毛利和已知业务变化，不必填满 150 个缺失项目。
只完成案例范围时，整体报告继续用“暂定样本＋已核查案例”的表述；案例数量不能代表全部历史已核实。


## Why These Companies and Cases Were Selected

**42 家公司的依据：研究问题相关性、业务类型覆盖与免费披露可获取性。**
它们是手工建立的候选名单，面向主要经营实体商品销售、库存与销售关系值得研究的消费零售及相关品牌企业。
不是随机样本、不是全部美国零售上市公司，也没有按未来毛利或现金流结果筛选候选名单。
我们没有事先指定市值最低值、美国收入占比阈值或统一 GICS 代码，报告中不能事后声称使用了这些门槛。
美国上市／SEC 披露是数据入口，不表示企业只在美国经营；品牌、批发和其他业务占比仍需要年报核查。

五个研究分组是研究者定义的工作分组，不是经过正式 GICS 或 NAICS 认证的分类：
- Apparel, Footwear, and Off-Price Retail：23 家；比较品牌服装、鞋类与折扣零售。
- Department Stores：3 家；比较百货模式，行业推断需谨慎。
- Home Furnishings, Furniture, and Home Improvement：7 家；比较家具、家居与家装，但产品周转周期差异较大。
- Sporting Goods, Electronics, and Other Merchandise：6 家；扩大商品类型覆盖，不能视为单一同质行业。
- Beauty and Personal Care：3 家；作为美容个护对照，样本很少，且部分商品具有较强经常性需求。

研究范围更准确的表述是 **selected merchandise retailers and consumer brands**。
若最后仍称 discretionary retail，应解释美容个护、专业客户销售等边界；尚未完成逐家正式行业资格核验。
零售库存研究已有先例：Gaur、Fisher 和 Raman（2005）研究零售企业库存周转与毛利、资本强度及销售意外的关系；这支持业务问题的重要性，但不能直接证明本项目的下一年预测假设。
参考：[原始研究](https://pubsonline.informs.org/doi/10.1287/mnsc.1040.0298)、[Census 零售业范围](https://www2.census.gov/econ2007/Reference_materials/htm%20files/ec44gsdesc.htm)、[SEC 数据接口](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)。

**7 个案例的依据：Module 11 当前信号筛选。**
每个分组选一个库存增长差最大的公司年度（5 个）；再选接近正差中位数的一条（1 个）与差最小的一条（1 个）。
这里的“最大”是在合格配对的并集内，不是在全部历史记录中；配对可用性仍受结果数据是否存在影响。
没有根据下一年表现好坏挑案例。当前信号极端、覆盖各组、含一个典型正差及一个反向对照，是选择目的。
- AKA 2021-12-31、KSS 2013-02-02、ARHS 2021-12-31、SPWH 2022-01-29、ULTA 2013-02-02：各组最大差。
- URBN 2011-01-31：接近正差中位数。
- SPWH 2021-01-30：最小差。
7 个公司年度来自 6 家公司；SPWH 的两条不是重复公司年度。案例仅帮助解释机制，不能估计行业发生率。

最终纳入依据：公司业务符合研究定义；身份对应正确；美元金额与年度范围可核实；每个结果至少 5 个可用年度；年度连续；避开已知不可比业务断点。
毛利率与现金流各自审核、各自筛样本，不因缺少毛利而删除本来可用的现金流记录。


## Cell 1: Set Up the Review Paths and Sample Mode

路径沿用 PROJECT_ROOT，实际项目目录为 `[project root]`。若打印不对，在导入后直接设置这个绝对路径。
REVIEW_DIR 专门存放可编辑审核文件；MODULE12_OUTPUT_DIR 存放本次重建结果。
两个 mkdir 只创建缺少的目录，不清空旧文件。
REPORT_SAMPLE 是字符串开关。先保留 provisional，让你看到未审核与审核决定如何影响样本；不要通过改变模式寻找显著结果。
MIN_ANNUAL_ROWS=5 延续原年度覆盖门槛，指年度数，不是最终配对数。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from IPython.display import display

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
REVIEW_DIR = PROCESSED_DIR / "module12_manual_review"
MODULE12_OUTPUT_DIR = PROCESSED_DIR / "module12_reviewed"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
MODULE12_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_SAMPLE = "provisional"
MIN_ANNUAL_ROWS = 5
assert REPORT_SAMPLE in ["provisional", "reviewed_only"], "Unknown sample mode."
print("Project directory:", PROJECT_ROOT)
print("Review files:", REVIEW_DIR)
print("Sample mode:", REPORT_SAMPLE)


## Cell 2: Create Four Editable Review Files Once

annual_source 读取 Module 9 的完整 656 行年度表，保留 DDS 修正、13 条桥接证据及原始金额。
不能只读已筛掉年份的研究样本，否则新增或排除记录之后，前后年度可能错接。
company_source、metric_source、case_source 分别读取 42 家公司、150 个待核查指标和 7 个案例的现有人工表；已有备注复制进来。

创建四份新工作表：
- company_review.csv：公司身份、工作分组和资格。新增 eligibility_review_status、eligibility_evidence_url、eligibility_notes。
- annual_scope_review.csv：逐年度范围与口径。新增 scope_status、scope_evidence_url、scope_notes；保留原有 scope_reason 供参考。
- metric_review.csv：缺失成本／毛利的金额审核。新增 reviewed_value_basis，区分 reported / derived。
- case_review.csv：沿用 Module 11 案例，记录库存驱动因素、未来比率变化原因及 retain / exclude_pair 决定。

年度表会提供上一期、本期、下一期 SEC 文件目录链接，以及 related_to_selected_case 标记。
先查相关年度，可以核对“上一期金额 → 本期增长 → 下一期结果”的完整链条；只看本期和下一期仍不够验证增长率。
这些 URL 是目录，不是已阅读的证据：进去找到 10-K 正文或报表文件，再把具体链接与页码／章节记入审核列。
如果某年缺少前后期，链接为空正常。你不需要用最新报告替代当年报告。

tables 是文件名到 DataFrame 的字典；for 循环检查文件是否已存在。不存在才 to_csv，因此不会覆盖人工填写。
如果未来更新了数据缓存，本模块不会自动把旧审核当作新年度审核；Cell 3 会检查键是否一致。


In [ ]:
annual_source = pd.read_csv(PROCESSED_DIR / "expanded_module9_annual_audit.csv", parse_dates=["period_start", "period_end", "filed"])
annual_source = annual_source.sort_values(["ticker", "period_end"]).reset_index(drop=True)
company_source = pd.read_csv(PROCESSED_DIR / "candidate_scope_review.csv", dtype={"cik10": "string"}, keep_default_na=False)
metric_source = pd.read_csv(PROCESSED_DIR / "expanded_metric_review_decisions.csv", keep_default_na=False)
case_source = pd.read_csv(PROCESSED_DIR / "expanded_module11_case_review.csv", dtype={"cik10": "string"}, keep_default_na=False)
company_source["eligibility_review_status"] = "pending"
company_source["eligibility_evidence_url"] = ""
company_source["eligibility_notes"] = ""
metric_source["reviewed_value_basis"] = ""
case_source["suggested_action"] = case_source["suggested_action"].replace("", "pending")
annual_review = annual_source[["ticker", "period_start", "period_end", "accn", "scope_reason", "known_scope_break", "historical_scope_hold"]].copy()
cik_lookup = company_source.set_index("ticker")["cik10"]
annual_review["current_filing_url"] = (
    "https://www.sec.gov/Archives/edgar/data/" + annual_review["ticker"].map(cik_lookup).astype(int).astype(str)
    + "/" + annual_review["accn"].str.replace("-", "", regex=False) + "/"
)
for direction, step in [("prev", 1), ("next", -1)]:
    annual_review[direction + "_period_end"] = annual_review.groupby("ticker")["period_end"].shift(step)
    annual_review[direction + "_filing_url"] = annual_review.groupby("ticker")["current_filing_url"].shift(step)
annual_review["related_to_selected_case"] = False
for _, case in case_source.iterrows():
    selected = annual_review["ticker"].eq(case["ticker"]) & annual_review["period_end"].eq(pd.Timestamp(case["period_end"]))
    dates = annual_review.loc[selected, ["prev_period_end", "period_end", "next_period_end"]].iloc[0]
    related = annual_review["ticker"].eq(case["ticker"]) & annual_review["period_end"].isin(dates)
    annual_review.loc[related, "related_to_selected_case"] = True
annual_review["scope_status"] = "pending"
annual_review["scope_evidence_url"] = ""
annual_review["scope_notes"] = ""
annual_review = annual_review.sort_values(["related_to_selected_case", "period_end"], ascending=[False, False])
tables = {"company_review.csv": company_source, "annual_scope_review.csv": annual_review, "metric_review.csv": metric_source, "case_review.csv": case_source}
for filename, table in tables.items():
    path = REVIEW_DIR / filename
    if not path.exists():
        table.to_csv(path, index=False, date_format="%Y-%m-%d")
    print(filename, "| rows:", len(table), "| existing notes retained")
display(annual_review.loc[annual_review["related_to_selected_case"], ["ticker", "period_end", "current_filing_url"]])


## Manual Review: What to Read and What to Enter

第一次先跑完确认系统正常。之后，在 JupyterLab 文件列表进入 `work/data/processed/module12_manual_review/`，打开或下载 CSV 编辑。
可以用表格软件，但 ticker、CIK、日期和 accn 必须保持文本格式，不要让前导零丢失或日期改格式；保留 UTF-8 CSV、列名和每条记录的键。
审核状态、备注与证据摘要请用英文填写；中文仅用于本 Guide 的教学说明。当前 7 个案例共涉及 19 条不重复的年度记录，已优先标记。
只修改下面指定的审核列，不改原始 ticker / period_end / accn / metric 或已有金额列。
如果使用文本编辑器，备注里出现逗号要放在双引号里；使用表格软件保存 CSV 会自动处理引号。

**公司资格：** 从 10-K 的 Business、收入分部和财务报表确认身份、主要商品销售模式以及品牌／批发／其他业务。
符合本项目范围：eligibility_review_status=reviewed，include_decision=include；不符合：reviewed / exclude。
两种决定都必须填写 eligibility_evidence_url 与 eligibility_notes。
不要把“美国上市”写成“收入主要来自美国”；没有事先设置纯零售收入比例门槛，就按业务描述判断并解释边界。
原 accounting_review_status 保留原状态；公司资格通过不等于全部历史会计审核完成。

**年度口径：** scope_status 使用 pending / reviewed / exclude。
reviewed 表示本年度拟用金额的单位、日期、实体范围及业务变化已核对，scope_evidence_url 与 scope_notes 必须填写。
应检查 Income / Operations、Balance Sheets、Cash Flows、Inventory、Business Combinations、Discontinued Operations 附注。
明确不可比则填 exclude 和理由，会影响涉及该年的上期增长、本期信号或下一期目标。
既有 known_scope_break 与 BBY 暂缓规则仍然保留；把状态改为 reviewed 不会自动撤销它们。
成本缺失而毛利已披露，可以说明并保留缺失；不能因缺一项就声称完整报表已核对。
如果无法确定则 pending，不填 reviewed。金额匹配、桥接差为零均不能代替人工核查。

**指标补值：** review_status 使用 pending / approved / rejected。
approved 必须填 reviewed_value、reviewed_value_basis（reported / derived）、evidence_url、review_notes。
金额统一 USD：thousands × 1,000，millions × 1,000,000；成本不能为负，毛利允许为负。
derived 只在公式、收入分母和毛利前费用全部确认后使用；proposed_value 只是旧建议，不自动采用。
本模块只补原来缺失的 cost_of_sales / gross_profit；不能用这个表改收入、库存、资产、现金流、日期或已存在金额。
若发现这些错误，先在上游有来源的步骤修正并重跑 Module 9，再同步新的审核文件；不能仅写备注就继续用旧金额。
新发现的额外毛利前费用也要在上游证据表登记，不用补值把桥接差硬凑成零。

**案例：** review_status 使用 pending / reviewed；suggested_action 使用 pending / retain / exclude_pair。
reviewed 要填写 current_evidence_url、next_evidence_url、units_and_scope_notes、revenue_and_inventory_drivers、next_margin_and_ocf_drivers、review_notes。
记录报表章节／页码和实际单位；管理层解释、报表事实与自己的推断要区分，不能把相关性写成因果。
retain 表示没有因案例核查新增排除理由，并非证明因果机制；exclude_pair 只排除这一条本期信号到下一期结果的配对。
若整个年度不可比，应同时在年度表设置 exclude，这样其他依赖该年度的配对也会受影响。
读过案例不会自动批准公司资格或三个年度；这三类审核的作用不同。

可在最终报告直接采用的英文范围说明：
> The candidate universe was manually assembled to cover selected merchandise retailers and consumer brands across five researcher-defined business groups. The initial 42-company list was not a census or a random sample. Final inclusion was assessed using business relevance, financial-data availability, annual continuity, and documented accounting-scope decisions. Cases were selected using contemporaneous inventory-growth gaps among available research pairs, rather than subsequent outcome direction. Results are retrospective associations and do not establish causal effects or validated predictive performance.


## Cell 3: Reload and Check the Human Decisions

重新读审核 CSV，才能得到磁盘上刚保存的备注。keep_default_na=False 让空备注保持空字符串，便于判断有没有填写。
只有状态值、批准金额和证据缺少这些会直接造成误用的项目才检查；不要求全部 pending 项目必须完成。
键与原始记录匹配是必要检查：否则一份旧 CSV 可能把审核决定误用到新版本财报。
approved 的 reviewed_value 转成数字，np.isfinite 排除空值、文本和无穷大；金额 0 可以合法，不能用 truthiness 判断。

如果报错，修改指出的审核记录后重跑本格；不要删除 assert 或把 pending 全改 approved 绕过。
当前全部 pending 时本格应顺利通过，并打印 reviewed 公司 0、reviewed 年度 0、approved 指标 0、reviewed 案例 0。


In [ ]:
company_review = pd.read_csv(REVIEW_DIR / "company_review.csv", dtype={"cik10": "string"}, keep_default_na=False)
scope_review = pd.read_csv(REVIEW_DIR / "annual_scope_review.csv", parse_dates=["period_end"], keep_default_na=False)
metric_review = pd.read_csv(REVIEW_DIR / "metric_review.csv", parse_dates=["period_end"], keep_default_na=False)
case_review = pd.read_csv(REVIEW_DIR / "case_review.csv", dtype={"cik10": "string"}, parse_dates=["period_end", "next_period_end"], keep_default_na=False)
assert company_review["eligibility_review_status"].isin(["pending", "reviewed"]).all(), "Invalid company review status."
assert company_review["include_decision"].isin(["pending", "include", "exclude"]).all(), "Invalid inclusion decision."
assert scope_review["scope_status"].isin(["pending", "reviewed", "exclude"]).all(), "Invalid annual scope status."
assert metric_review["review_status"].isin(["pending", "approved", "rejected"]).all(), "Invalid metric review status."
assert case_review["review_status"].isin(["pending", "reviewed"]).all(), "Invalid case review status."
assert case_review["suggested_action"].isin(["pending", "retain", "exclude_pair"]).all(), "Invalid case action."
assert len(company_review) == len(company_source), "Company list changed; reconcile review keys."
identity = company_review.merge(company_source[["ticker", "cik10", "subgroup"]], on=["ticker", "cik10", "subgroup"], validate="one_to_one", indicator=True)
assert len(identity) == len(company_source), "Company identity or group changed; reconcile review keys."
for review, source, keys in [
    (scope_review, annual_source, ["ticker", "period_end", "accn"]),
    (metric_review, metric_source.assign(period_end=pd.to_datetime(metric_source["period_end"])), ["ticker", "period_end", "accn", "metric"]),
    (case_review, case_source.assign(period_end=pd.to_datetime(case_source["period_end"]), next_period_end=pd.to_datetime(case_source["next_period_end"])), ["ticker", "period_end", "accn", "next_period_end", "next_accn"]),
]:
    matched = review.merge(source[keys], on=keys, how="left", validate="one_to_one", indicator=True)
    assert len(review) == len(source) and matched["_merge"].eq("both").all(), "Review keys changed; reconcile them before applying decisions."
for table, selected, columns in [
    (company_review, company_review["eligibility_review_status"].eq("reviewed") | company_review["include_decision"].ne("pending"), ["eligibility_evidence_url", "eligibility_notes"]),
    (scope_review, scope_review["scope_status"].ne("pending"), ["scope_evidence_url", "scope_notes"]),
    (metric_review, metric_review["review_status"].eq("approved"), ["evidence_url", "review_notes"]),
    (case_review, case_review["review_status"].eq("reviewed"), ["current_evidence_url", "next_evidence_url", "units_and_scope_notes", "revenue_and_inventory_drivers", "next_margin_and_ocf_drivers", "review_notes"]),
]:
    for column in columns:
        assert table.loc[selected, column].str.strip().ne("").all(), "Reviewed decisions need evidence and notes: " + column
decided = company_review["include_decision"].ne("pending")
assert company_review.loc[decided, "eligibility_review_status"].eq("reviewed").all(), "Company decisions need a completed eligibility review."
assert company_review.loc[company_review["eligibility_review_status"].eq("reviewed"), "include_decision"].ne("pending").all(), "Reviewed companies need an inclusion decision."
assert case_review.loc[case_review["review_status"].eq("reviewed"), "suggested_action"].isin(["retain", "exclude_pair"]).all(), "Reviewed cases need an action."
assert case_review.loc[case_review["review_status"].eq("pending"), "suggested_action"].eq("pending").all(), "Case actions require completed evidence review."
approved = metric_review.loc[metric_review["review_status"].eq("approved")].copy()
approved["reviewed_value"] = pd.to_numeric(approved["reviewed_value"], errors="coerce")
assert np.isfinite(approved["reviewed_value"].to_numpy(dtype=float)).all(), "Approved values must be finite USD amounts."
assert approved["reviewed_value_basis"].isin(["reported", "derived"]).all(), "Approved values need a reported or derived basis."
assert approved.loc[approved["metric"].eq("cost_of_sales"), "reviewed_value"].ge(0).all(), "Cost of sales must be nonnegative."
print("Reviewed companies:", int(company_review["eligibility_review_status"].eq("reviewed").sum()))
print("Reviewed annual rows:", int(scope_review["scope_status"].eq("reviewed").sum()))
print("Approved metric values:", len(approved))
print("Reviewed cases:", int(case_review["review_status"].eq("reviewed").sum()))


## Cell 4: Apply Approved Missing Values and Annual Scope Decisions

从 annual_source.copy() 重新开始，所以重复运行不会累积补值；original_ 两列保留补值前金额。
for / iterrows 每次取一条 approved 记录，match 同时检查股票、年度和 accession，避免用错年度或文件。
只允许补空值，existing_values.notna() 会阻止把已存在金额偷偷替换掉。
applied_rows 保存具体采用了什么、来自哪里、是 reported 还是 derived；被 rejected 的记录不写入金额。
derived_fields 原本是推导字段的数量，批准一个新的 derived 补值就加 1，不把它改成布尔状态。

随后合并年度 scope_status，把公司表的 include_decision 重新 map 回每一行。
provisional 中 pending 仍可进入，exclude 不可进入；reviewed_only 中公司必须 include / reviewed，年度必须 reviewed。
row_allowed 只控制进入分析，完整年度行不删除，保留后面 shift 的时间轴。
重新计算桥接差、基础金额条件、比率与至少 5 年覆盖，避免用旧的 Module 8 年度数量把补值或排除漏掉。
原 DDS 分析收入、额外毛利前费用、已知断点和 BBY hold 保持。只有明确的非零桥接差被挡住；没有差额不是已审核证明。


In [ ]:
annual = annual_source.copy()
annual["original_cost_of_sales"] = annual["cost_of_sales"]
annual["original_gross_profit"] = annual["gross_profit"]
annual["manual_metrics_applied"] = ""
applied_rows = []
for _, decision in approved.iterrows():
    metric = decision["metric"]
    assert metric in ["cost_of_sales", "gross_profit"], "This module only fills missing cost or gross profit."
    match = annual["ticker"].eq(decision["ticker"]) & annual["period_end"].eq(decision["period_end"]) & annual["accn"].eq(decision["accn"])
    assert match.sum() == 1 and annual.loc[match, metric].isna().all(), "Approved values may only fill a matching missing field."
    annual.loc[match, metric] = decision["reviewed_value"]
    annual.loc[match, "manual_metrics_applied"] += metric + ";"
    if decision["reviewed_value_basis"] == "derived":
        annual.loc[match, "derived_fields"] += 1
    applied_rows.append(decision[["ticker", "period_end", "accn", "metric", "reviewed_value", "reviewed_value_basis", "evidence_url", "review_notes"]].to_dict())
applied_values = pd.DataFrame(applied_rows, columns=["ticker", "period_end", "accn", "metric", "reviewed_value", "reviewed_value_basis", "evidence_url", "review_notes"])
annual = annual.merge(scope_review[["ticker", "period_end", "accn", "scope_status", "scope_evidence_url", "scope_notes"]], on=["ticker", "period_end", "accn"], validate="one_to_one")
company_lookup = company_review.set_index("ticker")
annual["include_decision"] = annual["ticker"].map(company_lookup["include_decision"])
annual["eligibility_review_status"] = annual["ticker"].map(company_lookup["eligibility_review_status"])
company_allowed = annual["include_decision"].ne("exclude")
annual["row_allowed"] = annual["scope_status"].ne("exclude")
if REPORT_SAMPLE == "reviewed_only":
    company_allowed = annual["include_decision"].eq("include") & annual["eligibility_review_status"].eq("reviewed")
    annual["row_allowed"] = annual["scope_status"].eq("reviewed")
annual["remaining_bridge_gap"] = annual["revenue_for_analysis"] - annual["cost_of_sales"] - annual["extra_gross_profit_charge"].fillna(0) - annual["gross_profit"]
annual["bridge_unresolved"] = annual["remaining_bridge_gap"].abs() > 1
annual["numeric_base_values_ok"] = annual["revenue_for_analysis"].gt(0) & annual["total_assets"].gt(0) & annual["inventory"].ge(0) & annual["inventory"].le(annual["total_assets"])
annual["base_values_ok"] = annual["numeric_base_values_ok"] & annual["row_allowed"]
annual["gm_row_usable"] = annual["base_values_ok"] & annual["gross_profit"].notna() & ~annual["bridge_unresolved"]
annual["ocf_row_usable"] = annual["base_values_ok"] & annual["operating_cash_flow"].notna()
annual["gross_margin"] = annual["gross_profit"] / annual["revenue_for_analysis"]
annual["ocf_margin"] = annual["operating_cash_flow"] / annual["revenue_for_analysis"]
annual.loc[~annual["gm_row_usable"], "gross_margin"] = float("nan")
annual.loc[~annual["ocf_row_usable"], "ocf_margin"] = float("nan")
coverage = annual.groupby("ticker")[["gm_row_usable", "ocf_row_usable"]].sum()
annual["gm_coverage_ok"] = annual["ticker"].map(coverage["gm_row_usable"]).ge(MIN_ANNUAL_ROWS) & company_allowed
annual["ocf_coverage_ok"] = annual["ticker"].map(coverage["ocf_row_usable"]).ge(MIN_ANNUAL_ROWS) & company_allowed
print("Applied missing values:", len(applied_values))
print("Annual rows retained for the timeline:", len(annual))


## Cell 5: Rebuild Signals and Outcomes on the Complete Timeline

这一格复用 Module 9 的时间配对规则，较长是因为需要把上期、本期、下一期分别检查，并非加入新的复杂模型。
先 sort_values，再 groupby / shift；不能先删除 excluded / pending 年份。
Cell 4 已把年度决定并入 base_values_ok，因此上期、当前期、下一期任一被排除，都不能继续形成相应的合格配对。
公司排除通过 coverage_ok 生效；案例 exclude_pair 另外按 ticker / period_end 定位，只排该条配对。

沿用 350–380 天终点差、下一期 start=本期 end+1 天、提交日早于目标终点、已知范围断点与 BBY hold。
重新算增长、毛利率变化、现金流比率变化，并清空不合格标签；不复制旧的最终结果列。
period_length 标记和提交时间差继续保留。人工审核不能把 filed 代理时间自动变成已验证的 point-in-time 数据。
这一格的毛利／现金流暂定数量在没有人工修改时应分别为 401 / 490。


In [ ]:
annual = annual.sort_values(["ticker", "period_end"]).reset_index(drop=True)
paired = annual.copy()
for column in [
    "period_end", "period_days", "revenue_for_analysis", "inventory",
    "base_values_ok", "known_scope_break", "historical_scope_hold",
]:
    paired["prev_" + column] = annual.groupby("ticker")[column].shift(1)
for column in [
    "period_start", "period_end", "period_days", "filed", "accn", "gross_margin", "ocf_margin",
    "gm_row_usable", "ocf_row_usable", "known_scope_break", "historical_scope_hold",
]:
    paired["next_" + column] = annual.groupby("ticker")[column].shift(-1)
previous_end_gap = (paired["period_end"] - paired["prev_period_end"]).dt.days
previous_start_gap = (paired["period_start"] - paired["prev_period_end"]).dt.days
paired["consecutive_previous_year"] = previous_end_gap.between(350, 380) & previous_start_gap.eq(1)
paired["feature_usable"] = (
    paired["consecutive_previous_year"] & paired["base_values_ok"] & paired["prev_base_values_ok"].eq(True)
    & paired["prev_revenue_for_analysis"].gt(0) & paired["prev_inventory"].gt(0)
    & ~paired["known_scope_break"] & paired["prev_known_scope_break"].eq(False)
    & ~paired["historical_scope_hold"] & paired["prev_historical_scope_hold"].eq(False)
)
paired["revenue_growth"] = paired["revenue_for_analysis"] / paired["prev_revenue_for_analysis"] - 1
paired["inventory_growth"] = paired["inventory"] / paired["prev_inventory"] - 1
paired["inventory_sales_gap"] = paired["inventory_growth"] - paired["revenue_growth"]
paired.loc[~paired["feature_usable"], ["revenue_growth", "inventory_growth", "inventory_sales_gap"]] = float("nan")
paired["inventory_outpaced_sales"] = paired["inventory_sales_gap"].gt(0).astype("boolean")
paired.loc[~paired["feature_usable"], "inventory_outpaced_sales"] = pd.NA
next_end_gap = (paired["next_period_end"] - paired["period_end"]).dt.days
next_start_gap = (paired["next_period_start"] - paired["period_end"]).dt.days
paired["consecutive_next_year"] = next_end_gap.between(350, 380) & next_start_gap.eq(1)
paired["signal_available_date"] = paired["filed"]
paired["filed_time_ok"] = (paired["filed"] >= paired["period_end"]) & (paired["filed"] < paired["next_period_end"])
excluded_case_keys = case_review.loc[case_review["review_status"].eq("reviewed") & case_review["suggested_action"].eq("exclude_pair"), ["ticker", "period_end"]]
paired["case_pair_excluded"] = False
for _, key in excluded_case_keys.iterrows():
    match = paired["ticker"].eq(key["ticker"]) & paired["period_end"].eq(key["period_end"])
    paired.loc[match, "case_pair_excluded"] = True
common_pair_ok = (
    paired["feature_usable"] & paired["consecutive_next_year"] & paired["filed_time_ok"] & ~paired["case_pair_excluded"]
    & paired["next_known_scope_break"].eq(False) & paired["next_historical_scope_hold"].eq(False)
)
paired["gm_pair_usable"] = common_pair_ok & paired["gm_coverage_ok"] & paired["gm_row_usable"] & paired["next_gm_row_usable"].eq(True)
paired["ocf_pair_usable"] = common_pair_ok & paired["ocf_coverage_ok"] & paired["ocf_row_usable"] & paired["next_ocf_row_usable"].eq(True)
print("Usable inventory feature rows:", int(paired["feature_usable"].sum()))
paired["next_gross_margin_change"] = paired["next_gross_margin"] - paired["gross_margin"]
paired["next_ocf_margin_change"] = paired["next_ocf_margin"] - paired["ocf_margin"]
for outcome, label, usable in [
    ("next_gross_margin_change", "gross_margin_declined", "gm_pair_usable"),
    ("next_ocf_margin_change", "ocf_margin_declined", "ocf_pair_usable"),
]:
    paired.loc[~paired[usable], outcome] = float("nan")
    paired[label] = paired[outcome].lt(0).astype("boolean")
    paired.loc[~paired[usable], label] = pd.NA
paired["available_before_target_start"] = paired["filed"] < paired["next_period_start"]
paired["days_after_target_start"] = (paired["filed"] - paired["next_period_start"]).dt.days
paired["period_length_changed"] = (paired["period_days"] - paired["prev_period_days"]).abs() > 3
paired["pair_period_length_changed"] = paired["period_length_changed"] | ((paired["next_period_days"] - paired["period_days"]).abs() > 3)
gm_pairs = paired.loc[paired["gm_pair_usable"]].copy()
ocf_pairs = paired.loc[paired["ocf_pair_usable"]].copy()
pair_summary = paired.groupby(["ticker", "subgroup"], sort=False)[["gm_pair_usable", "ocf_pair_usable"]].sum()
pair_summary = pair_summary.rename(columns={"gm_pair_usable": "gm_pairs", "ocf_pair_usable": "ocf_pairs"}).reset_index()

print("Rebuilt gross margin pairs:", len(gm_pairs))
print("Rebuilt OCF-to-revenue pairs:", len(ocf_pairs))


## Cell 6: Compare Sample Counts and Refresh the Seven Cases

baseline_settings 读取旧 Module 9 配对，作为修正前参照；新 gm_pairs / ocf_pairs 是刚重建的结果。
before_keys 和 after_keys 用 ticker / period_end 做左连接，统计移除与新增配对，不靠行号比较。
added_pairs 可能来自补回毛利；removed_pairs 可能来自公司、年度或案例排除。原因可以重叠，所以不把各规则失败计数相加当作独立删除数。

case_results 从完整 paired 取回最新信号与结果，再接上人工案例说明，避免输出旧案例文件中尚未更新的比率。
案例因审核被排除时，仍保留案例说明，但 pair_usable=False、相应变化为空。
这不会重新挑选新的极端案例，防止每次修正后不断换案例迎合结果。


In [ ]:
changes = []
for name, filename, table in [
    ("Gross margin", "expanded_gm_next_year_pairs.csv", gm_pairs),
    ("OCF-to-revenue", "expanded_ocf_next_year_pairs.csv", ocf_pairs),
]:
    before = pd.read_csv(PROCESSED_DIR / filename, parse_dates=["period_end"])
    before_keys = before[["ticker", "period_end"]]
    after_keys = table[["ticker", "period_end"]]
    removed = before_keys.merge(after_keys, on=["ticker", "period_end"], how="left", indicator=True)
    added = after_keys.merge(before_keys, on=["ticker", "period_end"], how="left", indicator=True)
    changes.append({"outcome": name, "before_pairs": len(before), "after_pairs": len(table), "after_companies": table["ticker"].nunique(), "removed_pairs": int(removed["_merge"].eq("left_only").sum()), "added_pairs": int(added["_merge"].eq("left_only").sum())})
sample_changes = pd.DataFrame(changes)
case_columns = ["ticker", "period_end", "selection_reason", "review_status", "suggested_action", "units_and_scope_notes", "revenue_and_inventory_drivers", "next_margin_and_ocf_drivers", "current_evidence_url", "next_evidence_url", "review_notes"]
case_results = case_review[case_columns].merge(paired, on=["ticker", "period_end"], validate="one_to_one")
case_results["inventory_gap_pp"] = case_results["inventory_sales_gap"] * 100
display(sample_changes)
display(case_results[["ticker", "period_end", "review_status", "suggested_action", "inventory_gap_pp", "gm_pair_usable", "ocf_pair_usable", "next_gross_margin_change", "next_ocf_margin_change"]])


## Cell 7: Summarize What Has Actually Been Reviewed

审核进度与样本量必须分开：读过 7 个案例不等于全部公司历史通过审核。
paired 的 scope_status 是本期；另外把年度状态 shift(1) 与 shift(-1)，检查上期和下一期。
company_eligibility_verified 要求 eligibility_review_status=reviewed 且 include_decision=include。
three_year_scope_reviewed 要求三个年度都 reviewed；同时满足时标为 review_complete_pair。
“已审核配对数量”只在真正可用的各结果样本内统计，不把不合格年度算进去。

report_readiness 还列出期长变化、目标开始前可用数量和缺失成本／毛利。
成本缺失不一定挡住已直接披露毛利的样本。没有补值记录也正常，说明本次没有批准补值。
point_in_time_verified / prediction_validated / causal_identification 都保持 False；人工会计审核不能替代这些研究设计。
reviewed_only 模式即便有少量样本，也需要足够公司和年度才可以重新拟合 Module 11，不自动宣布模型适用。


In [ ]:
paired["company_eligibility_verified"] = paired["eligibility_review_status"].eq("reviewed") & paired["include_decision"].eq("include")
paired["three_year_scope_reviewed"] = (
    paired["scope_status"].eq("reviewed")
    & annual.groupby("ticker")["scope_status"].shift(1).eq("reviewed")
    & annual.groupby("ticker")["scope_status"].shift(-1).eq("reviewed")
)
paired["review_complete_pair"] = paired["company_eligibility_verified"] & paired["three_year_scope_reviewed"]
gm_pairs = paired.loc[paired["gm_pair_usable"]].copy()
ocf_pairs = paired.loc[paired["ocf_pair_usable"]].copy()
readiness_rows = []
for name, table in [("Gross margin", gm_pairs), ("OCF-to-revenue", ocf_pairs)]:
    readiness_rows.append({"outcome": name, "pairs": len(table), "companies": table["ticker"].nunique(), "review_complete_pairs": int(table["review_complete_pair"].sum()), "pending_review_pairs": int((~table["review_complete_pair"]).sum()), "before_target_start_pairs": int(table["available_before_target_start"].sum()), "different_length_pairs": int(table["pair_period_length_changed"].sum())})
report_readiness = pd.DataFrame(readiness_rows)
display(report_readiness)
display(annual[["cost_of_sales", "gross_profit"]].isna().sum().rename("Remaining Missing Annual Fields").to_frame())
print("Point-in-time verified:", False)
print("Prediction validated:", False)
if gm_pairs.empty or ocf_pairs.empty:
    print("At least one outcome sample is empty. Continue evidence review before fitting that outcome.")


## Cell 8: Save Reviewed Decisions and Hand Off to the Analysis Modules

outputs 保存新的完整年度、完整配对、两个研究样本、公司配对数、补值证据、样本变化、案例结果与审核进度。
candidate_scope_review.csv 保存 Module 12 的公司决定副本，供后续 Module 11 读取 CIK 与资格。
metadata 从原 Module 9 说明复制后更新数量和规则；accounting_scope_verified 只有非空样本的全部保留配对都满足公司资格＋三个年度审核时才为 True。
原始 metadata 的读取路径从 PROJECT_ROOT 明确构建，避免 Module 10/11 修改共享 PROCESSED_DIR 后读错文件夹；先读入 metadata，再保存结果。
即便 True，也表示“按记录的人工审核通过”，不代表独立审计或因果／预测验证。
sample_status 在未完成时为 post_review_provisional；完成保留配对审核时为 reviewed_retained_pairs。不是把全部 42 家都标为合格。
审核文件不会在这格重写，所以不会抹掉刚填写的 notes。

**审核有修改后，如何重跑 Module 10 和 11？**
不用重跑原 Module 9；它尚不读取这些 Module 12 工作表，重跑它会重新生成旧的基础样本。
先完成本模块 Cell 3–8，再在 Module 10 的 Cell 1 最后加入下面三行，然后顺序重跑 Module 10 全部 8 格：
```python
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed" / "module12_reviewed"
FIGURE_DIR = PROCESSED_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
```
在 Module 11 的 Cell 1 最后也加同样三行，顺序重跑 Module 11 全部 8 格。
新增结果和图会在 Module 12 子目录，之前 Module 10–11 的原结果与图保留，便于比较。
原 Module 10 的共同样本、百分位和 Module 11 的样本量会按新数据自然改变，不应强行要求仍为 401 / 490 / 399。
空样本不要拟合；很小的已审核子集不适合套全部公司／年份模型。可保留案例研究与暂定全样本分析，说明限制。

**如果暂未编辑任何人工文件：** 应显示 review_complete_pairs=0、applied values=0，仍保留 401 / 490 暂定配对。
此时完成的是审核系统搭建，尚未完成年报核查；不能直接进入“最终已审核结论”。
Module 13 将依据实际已核实的范围整理报告，仍未完成的部分明确列为限制。


In [ ]:
metadata_path = PROJECT_ROOT / "work" / "data" / "processed" / "expanded_module9_metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
outputs = {
    "expanded_module12_annual_audit.csv": annual,
    "expanded_module12_pairing_audit.csv": paired,
    "expanded_gm_next_year_pairs.csv": gm_pairs,
    "expanded_ocf_next_year_pairs.csv": ocf_pairs,
    "expanded_next_year_pair_counts.csv": pair_summary,
    "expanded_module12_applied_values.csv": applied_values,
    "expanded_module12_sample_changes.csv": sample_changes,
    "expanded_module12_case_results.csv": case_results,
    "expanded_module12_report_readiness.csv": report_readiness,
    "candidate_scope_review.csv": company_review,
}
for filename, table in outputs.items():
    table.to_csv(MODULE12_OUTPUT_DIR / filename, index=False, date_format="%Y-%m-%d")
retained_keys = pd.concat([gm_pairs, ocf_pairs]).drop_duplicates(["ticker", "period_end"])
scope_verified = not retained_keys.empty and retained_keys["review_complete_pair"].all()
metadata.update({
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "review_module": 12,
    "report_sample_mode": REPORT_SAMPLE, "sample_status": "reviewed_retained_pairs" if scope_verified else "post_review_provisional",
    "gm_companies": gm_pairs["ticker"].nunique(), "gm_pairs": len(gm_pairs), "ocf_companies": ocf_pairs["ticker"].nunique(), "ocf_pairs": len(ocf_pairs),
    "min_annual_coverage": MIN_ANNUAL_ROWS, "coverage_rule": "Recount usable annual rows after numeric and annual-scope decisions; company eligibility also required in reviewed-only mode",
    "manual_metric_decisions_applied": not applied_values.empty,
    "manual_review_decisions_applied": bool(company_review["eligibility_review_status"].eq("reviewed").any() or scope_review["scope_status"].ne("pending").any() or case_review["review_status"].eq("reviewed").any() or not applied_values.empty),
    "applied_metric_values": len(applied_values), "reviewed_annual_rows": int(scope_review["scope_status"].eq("reviewed").sum()), "reviewed_cases": int(case_review["review_status"].eq("reviewed").sum()),
    "accounting_scope_verified": bool(scope_verified), "point_in_time_verified": False, "prediction_validated": False, "causal_identification": False,
    "scope_rule": "Retain original known breaks and BBY hold; apply manual annual exclusions to all dependencies and documented case exclusions to individual pairs",
    "metric_rule": "Only approved missing cost/gross-profit values with recorded evidence are filled; original amounts preserved",
    "case_rule": "Keep Module 11 selected cases fixed; refresh computed results after review decisions",
    "review_directory": str(REVIEW_DIR),
})
for filename in ["expanded_module12_metadata.json", "expanded_module9_metadata.json"]:
    (MODULE12_OUTPUT_DIR / filename).write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Module 12 outputs saved in:", MODULE12_OUTPUT_DIR)
print("Retained-pair accounting review complete:", bool(scope_verified))
